In [ ]:
# Imports and constants
from pathlib import Path
import pandas as pd

# When implemented with python files it should be changed 
# to a path that is absolute at runtime
ROOT = Path.cwd().resolve()
while not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent

PATH_TO_DATA_FILES = ROOT / "data"

survey_columns = ["submission_id", "timestamp", "user_email", "rating", "comment_text", "region"]
user_metadata_columns = ["user_email", "full_name", "department", "country"]

# Load raw data and create copies

In [ ]:
survey_data = pd.read_csv(PATH_TO_DATA_FILES/"survey_results.csv")
user_metadata = pd.read_csv(PATH_TO_DATA_FILES/"user_metadata.csv")

In [ ]:
# Create staging copies
# For simplicity of implementation, if the data volume was larger or there were memory considerations the copying could be removed
stg_survey_results = survey_data.copy()
stg_user_metadata = user_metadata.copy()

print(f"Handling {len(stg_survey_results)} rows of survey data")
print(f"Handling {len(stg_user_metadata)} rows of user metadata")


## Data Cleaning of Survey Data

In [ ]:
# Remove exact duplicates
_length_before_removal = len(stg_survey_results)
stg_survey_results = stg_survey_results.drop_duplicates()
number_of_duplicates_removed = _length_before_removal - len(stg_survey_results)
print(f"Removed: {number_of_duplicates_removed} duplicates")

# Normalize rating; keep invalid as NA and flag it
stg_survey_results['rating'] = pd.to_numeric(stg_survey_results['rating'], errors='coerce')
stg_survey_results.loc[~stg_survey_results['rating'].between(1, 5), 'rating'] = None
print(f"Number of invalid ratings = {len(stg_survey_results.loc[stg_survey_results['rating'].isna()])}")

# Parse timestamp to datetime column keeping null values
stg_survey_results['submitted_at'] = pd.to_datetime(stg_survey_results['timestamp'], errors='coerce')
print(f"Number of invalid timestamps = {len(stg_survey_results.loc[stg_survey_results['submitted_at'].isna()])}")

# Normalize emails keeping null values
stg_survey_results['user_email'] = (
    stg_survey_results['user_email'].astype(str)
    .str.strip()
    .str.lower()
    .replace({'nan': None, 'None': None, '': None})
)
print(f"Number of invalid emails = {len(stg_survey_results.loc[stg_survey_results['user_email'].isna()])}")

# Drop all rows with at least one NaN value
_before = len(stg_survey_results)
stg_survey_results = stg_survey_results.dropna()
print(f"Dropped {_before - len(stg_survey_results)} rows because of NaN values")

## Data Cleaning of User Metadata

In [ ]:
# Standardize department (could be further enhanced combining strings with similar semantic meaning
# like Talent Acquision and HR)
stg_user_metadata['department'] = (
    stg_user_metadata['department'].astype(str)
    .str.strip()
    .str.title()
    .replace({'nan': None, 'None': None, '': None})
)
print(f"Number of rows with missing department = {len(stg_user_metadata.loc[stg_user_metadata['department'].isna()])}")

# Flag missing country
stg_user_metadata['country'] = (
    stg_user_metadata['country'].astype(str)
    .str.strip()
    .str.title()
    .replace({'nan': None, 'None': None, '': None})
)
print(f"Number of rows with missing country = {len(stg_user_metadata.loc[stg_user_metadata['country'].isna()])}")


# Normalize emails keeping null values
stg_user_metadata['user_email'] = (
    stg_user_metadata['user_email'].astype(str)
    .str.strip()
    .str.lower()
    .replace({'nan': None, 'None': None, '': None})
)
print(f"Number of invalid emails = {len(stg_user_metadata.loc[stg_user_metadata['user_email'].isna()])}")
stg_user_metadata = stg_user_metadata.dropna(subset=["user_email"])

# Create column with country and department suffixed to full name to only flag names with different emails when respondents have everything in common
stg_user_metadata['name_and_country_and_department'] = stg_user_metadata['full_name'].astype(str) + "_" + stg_user_metadata['country'].astype(str)  + "_" + stg_user_metadata['department'].astype(str)

# Count respondents with more than one email and flag those rows
email_counts = stg_user_metadata.groupby("name_and_country_and_department")["user_email"].transform("nunique")
stg_user_metadata['multiple_emails'] = email_counts > 1

print(f"There are {stg_user_metadata['multiple_emails'].sum()} respondents with two emails for the same combination of name, department and country")

# Drop all rows with at least one NaN value
_before = len(stg_user_metadata)
stg_user_metadata = stg_user_metadata.dropna()
print(f"Dropped {_before - len(stg_user_metadata)} rows because of NaN values")


## Analytics Fact Table

In [ ]:
# Join to user metadata (left join: preserve survey responses even if we have no user metadata)
fct_survey_feedback = stg_survey_results.copy()
fct_survey_feedback = fct_survey_feedback.merge(stg_user_metadata, on='user_email', how='left')
fct_survey_feedback = fct_survey_feedback.drop(columns=['name_and_country_and_department', 'multiple_emails', 'timestamp'])
if len(fct_survey_feedback.dropna(subset=user_metadata_columns)) < len(stg_survey_results):
    print(f"There were {len(stg_survey_results) - len(fct_survey_feedback.dropna(subset=user_metadata_columns))} surveys with missing metadata")
    print("Dropping these surveys")
    fct_survey_feedback = fct_survey_feedback.dropna(subset=user_metadata_columns)



## Aggregations

In [ ]:
# Included a count of the number of responses per group since it can be interesting for analysis
agg_by_department = (
    fct_survey_feedback.groupby('department')
    .agg(avg_rating=('rating','mean'), response_count=('submission_id','count'))
    .reset_index()
    .sort_values('response_count', ascending=False)
)

agg_by_region = (
    fct_survey_feedback.groupby('region')
    .agg(avg_rating=('rating','mean'), response_count=('submission_id','count'))
    .reset_index()
)

rating_distribution = (
    fct_survey_feedback.groupby(['department', 'rating'])
    .size()
    .reset_index(name="rating_count")
)



In [ ]:
# Save aggregations to csv

AGGREGATIONS_DIR = ROOT / 'aggregations'
AGGREGATIONS_DIR.mkdir(exist_ok=True)
agg_by_department.to_csv(AGGREGATIONS_DIR/'agg_by_department.csv', index=False)
agg_by_region.to_csv(AGGREGATIONS_DIR/'agg_by_region.csv', index=False)
rating_distribution.to_csv(AGGREGATIONS_DIR/'rating_distribution.csv', index=False)
